# INTERNAL REVIEW COPY - DO NOT PRESENT

> **Welcome!** This notebook is specifically tailored for **YOU (the researchers)**. 
> It is packed with in-depth comments, step-by-step logic breakdowns, and methodology 
> rationale. If you ever need to understand *why* a block of code is written the way it is, 
> read the markdown sections and code comments here!

---


# Philippine Real Estate: Predicting Property Prices

This notebook fits the two models of the study — **Random Forest Regression** and
**XGBoost Regression** — on the Philippine real-estate asking-price dataset and reports
the metrics that the Results & Discussion will quote.

It is written to be executed headlessly, and every claim it makes is guarded by a plain
`assert`. There are no test dependencies.

```
jupyter nbconvert --to notebook --execute --inplace notebook/price_prediction_pipeline.ipynb
```

**Note on Methodology:** To ensure a fair comparison between Random Forest and XGBoost, we are using fixed hyperparameters rather than dynamically tuning them. This guarantees both models are evaluated on an equal footing without inadvertently giving one an unfair advantage.

In [1]:
import gc
import json
import platform
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
import xgboost
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    r2_score,
    root_mean_squared_error,
)
from sklearn.model_selection import KFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, TargetEncoder
from xgboost import XGBRegressor

# Resolve the project root from the notebook's location, so the notebook runs the same
# whether nbconvert is invoked from the project root or from notebook/.
HERE = Path.cwd().resolve()
PROJECT = HERE if (HERE / "data" / "listings.csv").exists() else HERE.parent
DATA = PROJECT / "data" / "listings.csv"
MANIFEST = PROJECT / "dataset_manifest.json"
RESULTS = PROJECT / "results" / "metrics.json"
ARTIFACT = PROJECT / "artifacts" / "price_pipeline.joblib"
RESULTS.parent.mkdir(parents=True, exist_ok=True)
ARTIFACT.parent.mkdir(parents=True, exist_ok=True)

# ---- Locked constants (RESULTS_LOG.md sections 3 and 5) --------------------
GLOBAL_SEED = 42
SPLIT_SEED = 42
TEST_SIZE = 0.2
TARGET_ENCODER_FOLDS = 5
EXPECTED_ROWS_RAW = 241_496
EXPECTED_ROWS_DEDUP = 226_438
EXPECTED_COLUMNS = 25

TARGET = "price_php"
NUMERIC_FEATURES = ["sqm", "lot_area_m2", "bedrooms", "bathrooms", "lat", "lng"]
CATEGORICAL_FEATURES = ["city", "province", "region"]
ONEHOT_FEATURES = ["property_type"]
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES + ONEHOT_FEATURES

# Excluded on purpose. price_per_sqm is the target divided by sqm, so feeding it to the
# model is target leakage. district / barangay / property_subtype are unavailable at
# inference time, so a model using them could not actually be deployed.
LEAKAGE_COLUMNS = ["price_per_sqm"]
UNAVAILABLE_AT_INFERENCE = ["district", "barangay", "property_subtype"]

RAW_COLUMNS = [
    "canonical_id",
    "last_seen",
    TARGET,
    "price_per_sqm",
    *FEATURES,
]

# The spatial holdout is a single city held out entirely, not a random split. Naming
# the city explicitly is what makes it a *spatial* holdout.
SPATIAL_HOLDOUT_CITY = "Davao City"

print(f"python {platform.python_version()} | pandas {pd.__version__} | "
      f"sklearn {sklearn.__version__} | xgboost {xgboost.__version__}")
print(f"project root: {PROJECT}")
assert DATA.exists(), f"dataset not found: {DATA}"
assert MANIFEST.exists(), f"manifest not found: {MANIFEST}"

python 3.14.6 | pandas 3.0.5 | sklearn 1.9.0 | xgboost 3.4.1
project root: C:\Users\Lethe\Documents\Workspaces\General\General - Obsidian\Projects\IT20L_2026-2027_1STSEM-1STTERM\Deliverables\FINAL_PROJECT_1


## 1. Cleaning the Data (Deduplication)

The raw file has **241,496** rows but only **226,438** distinct `canonical_id` values:
15,058 surplus rows from relisted properties. Each `canonical_id` is observed several
times, with `last_seen` recording the most recent sighting.

Keeping the **latest `last_seen`** per `canonical_id` is therefore the first modelling
decision, and it happens before the train/test split — otherwise the same property would
appear on both sides of the split and inflate every metric.

In [2]:
def load_raw() -> pd.DataFrame:
    frame = pd.read_csv(DATA, usecols=RAW_COLUMNS, low_memory=False)
    assert len(frame) == EXPECTED_ROWS_RAW, (
        f"expected {EXPECTED_ROWS_RAW:,} raw rows, found {len(frame):,}. The dataset "
        f"has changed - re-check the SHA256 in dataset_manifest.json before continuing."
    )
    return frame


def deduplicate(frame: pd.DataFrame) -> pd.DataFrame:
    """Keep the most recent observation of each canonical_id.

    ``last_seen`` is parsed rather than sorted as text, and the sort is a stable mergesort
    with ``canonical_id`` as a tie-breaker, so the result does not depend on the input
    row order.
    """
    ordered = frame.assign(_last_seen=pd.to_datetime(frame["last_seen"], errors="coerce"))
    unparsed = int(ordered["_last_seen"].isna().sum())
    assert unparsed == 0, f"{unparsed} rows have an unparseable last_seen value"

    ordered = ordered.sort_values(
        ["_last_seen", "canonical_id"], kind="mergesort", ignore_index=True
    )
    deduped = ordered.drop_duplicates(subset="canonical_id", keep="last").reset_index(drop=True)

    # VERIFICATION 1: deduplicated row count == 226,438
    assert len(deduped) == EXPECTED_ROWS_DEDUP, (
        f"expected {EXPECTED_ROWS_DEDUP:,} deduplicated rows, found {len(deduped):,}"
    )
    assert deduped["canonical_id"].is_unique, "canonical_id is not unique after dedup"

    dropped = len(frame) - len(deduped)
    print(f"raw rows            {len(frame):>9,}")
    print(f"surplus rows dropped{dropped:>10,}")
    print(f"deduplicated rows   {len(deduped):>9,}")
    return deduped.drop(columns="_last_seen")


raw = load_raw()
data = deduplicate(raw)
del raw
data.head(3)

raw rows              241,496
surplus rows dropped    15,058
deduplicated rows     226,438


,canonical_id,property_type,price_php,sqm,lot_area_m2,price_per_sqm,bedrooms,bathrooms,city,province,region,lat,lng,last_seen
0,00ef7e5f1470355f,house,62000000.0,900.0,NaN,68888.89,NaN,9.0,City of Mandaluyong,"NCR, Second District",National Capital Region (NCR),14.578,121.041,2026-02-22
1,014fd1ffc8e76f86,lot,3500000.0,930.0,NaN,3763.44,NaN,NaN,Panglao,Bohol,Region VII (Central Visayas),9.579,123.745,2026-02-22
2,0190f73f057df9aa,lot,800000.0,100.0,NaN,8000.00,NaN,NaN,Naga Camarines Sur,NaN,NaN,13.616,123.241,2026-02-22


## 2. Feature Engineering & Preventing Data Leaks

One deterministic back-fill is applied: for **condominiums** `lot_area_m2` is missing for
82% of rows, but a condo's floor area *is* its lot area, so `sqm` is copied across. This
is a domain rule, not a statistic learned from data, so it is applied before splitting
without leaking anything.

Everything else is left as-is. In particular the out-of-domain values catalogued in
`RESULTS_LOG.md` §5.1 (a `sqm` of 45,000,000, 301 bedrooms) are **measured and reported,
not silently cleaned** — a predictor with an unbounded plausible input range is a real
limitation worth stating in the Discussion rather than hiding.

In [3]:
def build_features(frame: pd.DataFrame) -> pd.DataFrame:
    frame = frame.copy()

    # Condo lot area back-fill (RESULTS_LOG.md section 3.2).
    condo = frame["property_type"] == "condo"
    backfilled = condo & frame["lot_area_m2"].isna() & frame["sqm"].notna()
    frame.loc[backfilled, "lot_area_m2"] = frame.loc[backfilled, "sqm"]
    filled = int(backfilled.sum())
    print(f"condo lot_area_m2 back-filled from sqm: {filled:,} rows")
    assert not frame.loc[condo & frame["sqm"].notna(), "lot_area_m2"].isna().any(), (
        "condo rows with a known sqm still have a missing lot_area_m2"
    )

    X = frame[FEATURES].copy()

    # VERIFICATION 2: no target leakage columns in the feature matrix.
    offenders = sorted(set(X.columns) & ({TARGET} | set(LEAKAGE_COLUMNS)))
    assert not offenders, f"target leakage in the feature matrix: {offenders}"
    assert TARGET not in X.columns
    assert "price_per_sqm" not in X.columns
    for column in UNAVAILABLE_AT_INFERENCE:
        assert column not in X.columns, (
            f"{column} is unavailable at inference and must not be a feature"
        )
    assert list(X.columns) == FEATURES, "feature order drifted from the locked set"

    y = frame[TARGET].astype("float64")
    assert y.notna().all(), "missing target values"
    assert (y > 0).all(), "non-positive asking price in the target"
    return X, y


X, y = build_features(data)
print(f"\nfeatures {X.shape}, target {y.shape}")
print(f"target: min P{y.min():,.0f} | median P{y.median():,.0f} | mean P{y.mean():,.0f} "
      f"| max P{y.max():,.0f}")
X.head(3)

condo lot_area_m2 back-filled from sqm: 66,253 rows

features (226438, 10), target (226438,)
target: min P350,000 | median P12,060,000 | mean P41,241,101 | max P1,000,000,000


,sqm,lot_area_m2,bedrooms,bathrooms,lat,lng,city,province,region,property_type
0,900.0,NaN,NaN,9.0,14.578,121.041,City of Mandaluyong,"NCR, Second District",National Capital Region (NCR),house
1,930.0,NaN,NaN,NaN,9.579,123.745,Panglao,Bohol,Region VII (Central Visayas),lot
2,100.0,NaN,NaN,NaN,13.616,123.241,Naga Camarines Sur,NaN,NaN,lot


## 3. Preparing the Data for the Models

Two different treatments, because the two groups of columns mean different things.

**Numeric** (`sqm`, `lot_area_m2`, `bedrooms`, `bathrooms`, `lat`, `lng`) — median
imputation. The median is robust to the extreme values in §5.1, which a mean would chase.

**Categorical** (`city`, `province`, `region`) — most-frequent imputation, then
**target encoding** with internal 5-fold cross-fitting. One-hot encoding is not viable
here: there are 1,530 distinct `city` values, 337 of which appear exactly once, and median
asking price varies about **2,467-fold** across cities. A one-hot column for a city seen
once is nearly constant, and most-frequent imputation would silently merge hundreds of
distinct cities into one bucket. Target encoding instead gives each city its own
price signal.

The cross-fitting is what keeps this honest: a category's encoded value during training
comes from folds that excluded that row, so the model never sees a category's own target.

We use 5-fold cross-fitting here purely to prevent data leakage. This ensures a city's encoded value is calculated from other rows, meaning the model never accidentally sees a city's own price during training.
**One-hot** (`property_type`) — 4 categories, so one-hot is the right encoding.

**On the excluded columns.** `district`, `barangay` and `property_subtype` are dropped
because they are *text* fields that are mostly empty (77%, 83% and 69% missing) and would
need a lookup or geocoding step to supply at inference. `lat` and `lng` are kept because
they are numeric measurements a seller or portal already records on the listing form — no
resolution step is needed to obtain them. The distinction is field type and availability,
not location granularity: coordinates do resolve to a barangay, but they are present on
the listing whereas the barangay *string* usually is not.

`StandardScaler` is **not** used. Both models are tree ensembles, which are invariant to
feature scale, and the methodology states no standardisation is required.

In [4]:
def build_preprocessor(extra_numeric=()) -> ColumnTransformer:
    """Build the locked preprocessor.

    ``extra_numeric`` appends numeric columns to the imputed branch. It exists for the
    leakage audit in section 8, which must genuinely feed the estimator the extra column
    rather than assume it arrived.
    """
    numeric = [*NUMERIC_FEATURES, *extra_numeric]
    return ColumnTransformer(
        transformers=[
            ("num", SimpleImputer(strategy="median"), numeric),
            (
                "te",
                Pipeline(
                    steps=[
                        ("impute", SimpleImputer(strategy="most_frequent")),
                        (
                            "encode",
                            TargetEncoder(
                                target_type="continuous",
                                smooth="auto",
                                cv=KFold(
                                    TARGET_ENCODER_FOLDS,
                                    shuffle=True,
                                    random_state=GLOBAL_SEED,
                                ),
                            ),
                        ),
                    ]
                ),
                CATEGORICAL_FEATURES,
            ),
            (
                "onehot",
                Pipeline(
                    steps=[
                        ("impute", SimpleImputer(strategy="most_frequent")),
                        (
                            "encode",
                            OneHotEncoder(
                                handle_unknown="ignore", sparse_output=False
                            ),
                        ),
                    ]
                ),
                ONEHOT_FEATURES,
            ),
        ],
        remainder="drop",
        sparse_threshold=0,
    )


preprocessor = build_preprocessor()

# Every locked feature must be claimed by a transformer. ColumnTransformer drops anything
# it is not told about, with remainder="drop" and no warning, so a column added by mistake
# would vanish and the model would quietly train without it.
def assert_preprocessor_consumes(pre, expected_columns) -> None:
    consumed = set()
    for _, _, columns in pre.transformers:
        consumed.update(columns)
    unconsumed = sorted(set(expected_columns) - consumed)
    assert not unconsumed, (
        f"preprocessor silently drops {unconsumed}; the model would never see them and "
        f"the results would not reflect the intended feature set"
    )
    unexpected = sorted(consumed - set(expected_columns))
    assert not unexpected, f"preprocessor references columns not supplied: {unexpected}"


assert_preprocessor_consumes(preprocessor, FEATURES)

# Fit a throwaway copy on a small slice purely to learn the transformed column names and
# width. Kept as an explicit global so check_fitted_pipeline can assert the width does
# not change between fit and transform.
_probe = clone(preprocessor)
_probe.fit(X.iloc[:2000], y.iloc[:2000])
probe_names = list(_probe.get_feature_names_out())
EXPECTED_TRANSFORMED_WIDTH = len(probe_names)
print(f"transformed matrix width: {EXPECTED_TRANSFORMED_WIDTH}")
print(f"columns: {probe_names}")

transformed matrix width: 13
columns: ['num__sqm', 'num__lot_area_m2', 'num__bedrooms', 'num__bathrooms', 'num__lat', 'num__lng', 'te__city', 'te__province', 'te__region', 'onehot__property_type_commercial', 'onehot__property_type_condo', 'onehot__property_type_house', 'onehot__property_type_lot']


## 4. Setting Up Random Forest and XGBoost

These are **fixed, not searched**. The table below is the one reproduced in the
Methodology section.

The configurations are matched where matching is meaningful — the same `n_estimators`,
`random_state` and `n_jobs` — and differ where the algorithms genuinely differ. XGBoost
additionally gets a learning rate, row and column subsampling, and L2 regularisation,
because those are what its sequential boosting needs to avoid overfitting. Random Forest
instead gets `max_features=0.5` and a leaf-size floor, which is how bagging is
regularised.

**`max_depth` differs by design, 20 for Random Forest against 8 for XGBoost, and the
reason is stated here so it is not mistaken for a search result.** The two algorithms
reach depth differently. A Random Forest tree grows to its full depth in one step, so 20
levels are needed for the interactions within a single tree. An XGBoost tree is one
boosting round and is stacked on the previous rounds, so its effective depth is the sum
over rounds; the standard practice is therefore a shallow per-round tree, and 8 is
conventional rather than chosen. Giving XGBoost Random Forest's depth of 20 would give it
an effective depth of roughly 20 × 300, not a like-for-like comparison.

`max_features` is set **explicitly** because scikit-learn 1.9 now defaults it to `1.0`.
Leaving the default would give Random Forest a different search space from the one the
methodology describes.

In [5]:
MODEL_CONFIG = {
    "RandomForestRegression": {
        "estimator": "sklearn.ensemble.RandomForestRegressor",
        "n_estimators": 300,
        "max_depth": 20,
        "max_features": 0.5,
        "min_samples_leaf": 2,
        "random_state": GLOBAL_SEED,
        "n_jobs": -1,
    },
    "XGBoostRegression": {
        "estimator": "xgboost.XGBRegressor",
        "n_estimators": 300,
        "max_depth": 8,
        "learning_rate": 0.05,
        "subsample": 0.9,
        "colsample_bytree": 0.9,
        "reg_lambda": 1.0,
        "tree_method": "hist",
        "random_state": GLOBAL_SEED,
        "n_jobs": -1,
    },
}


def build_estimator(name: str):
    """Construct an estimator from the locked config table."""
    config = dict(MODEL_CONFIG[name])
    config.pop("estimator")
    if name == "RandomForestRegression":
        return RandomForestRegressor(**config)
    if name == "XGBoostRegression":
        return XGBRegressor(**config)
    raise KeyError(name)


config_table = pd.DataFrame(MODEL_CONFIG).T
print("Fixed hyperparameter configuration (no search, no early stopping)\n")
print(config_table.to_string())

# The two configurations must be comparable, not one secretly favoured by a search.
assert MODEL_CONFIG["RandomForestRegression"]["n_estimators"] == (
    MODEL_CONFIG["XGBoostRegression"]["n_estimators"]
), "n_estimators differ between the two configurations; the comparison is not symmetric"
assert MODEL_CONFIG["RandomForestRegression"]["random_state"] == (
    MODEL_CONFIG["XGBoostRegression"]["random_state"]
)
assert (
    RandomForestRegressor(n_estimators=2).max_features == 1.0
), "the max_features default changed; re-check the RESULTS_LOG.md section 4 note"
print("\nno early stopping: n_estimators is fixed for both models")

Fixed hyperparameter configuration (no search, no early stopping)

                                                     estimator n_estimators max_depth max_features min_samples_leaf random_state n_jobs learning_rate subsample colsample_bytree reg_lambda tree_method
RandomForestRegression  sklearn.ensemble.RandomForestRegressor          300        20          0.5                2           42     -1           NaN       NaN              NaN        NaN         NaN
XGBoostRegression                         xgboost.XGBRegressor          300         8          NaN              NaN           42     -1          0.05       0.9              0.9        1.0        hist

no early stopping: n_estimators is fixed for both models


## 5. How We Measure Success

Primary, reported in **pesos**: MAE, RMSE, R².

R² alone is not enough here. The target is extremely right-skewed — median ₱12,060,000
against a mean of ₱41,241,101 and a maximum of ₱1B — so R² is dominated by a handful of
luxury properties and a model can look respectable while being useless on a typical
listing. The additional metrics answer the questions a parent actually asks:

- **MAPE** — typical percentage error. Unstable when prices approach zero, reported for
    completeness.
- **MAE as % of median** — the error a *typical* listing sees, which the mean obscures.
- **Log-space R²** — R² computed on `log1p` of price and prediction, i.e. it scores
    *multiplicative* accuracy: the same prediction is penalised the same whether it is
    10% high on a ₱2M house or 10% high on a ₱200M house. It does **not** measure ranking
    or ordering, and it is not a percentile measure.
- **Calibration slope** — the least-squares slope of `log1p(true)` on `log1p(predicted)`.
    1.0 is a multiplicatively unbiased model; below 1.0 means systematic shrinkage toward
    the middle of the range.
- **Non-positive predictions** — a price cannot be zero or negative. Any such prediction
    is a defect, and because `log1p` is undefined there it dominates the log-space metric
    while barely moving the peso-scale ones. The count and the log-space result excluding
    those rows are both reported, so the effect is visible rather than hidden.

💡 **Researcher Tip:** Why log space? Because predicting ₱1 Million wrong on a ₱5M house is a big deal, but predicting ₱1 Million wrong on a ₱100M mansion is negligible. Log space forces the model to care about percentages (multiplicative error) rather than raw pesos!

In [6]:
def evaluate(y_true: pd.Series, y_pred: np.ndarray) -> dict:
    """Primary and supplementary metrics for one model on one holdout set."""
    y_true = np.asarray(y_true, dtype="float64")
    y_pred = np.asarray(y_pred, dtype="float64")

    mae = float(mean_absolute_error(y_true, y_pred))
    rmse = float(root_mean_squared_error(y_true, y_pred))
    r2 = float(r2_score(y_true, y_pred))
    mape_pct = float(mean_absolute_percentage_error(y_true, y_pred)) * 100
    median = float(np.median(y_true))
    mae_pct_of_median = mae / median * 100

    # A price is never zero or negative, so any such prediction is a defect. log1p is
    # undefined there, so these rows are counted and isolated rather than quietly
    # clipped away without comment.
    non_positive = y_pred <= 0
    n_non_positive = int(non_positive.sum())
    worst_non_positive = float(y_pred.min())

    # log1p needs non-negative input; clip rather than drop rows.
    log_true = np.log1p(np.clip(y_true, 0, None))
    log_pred = np.log1p(np.clip(y_pred, 0, None))
    log_r2 = float(r2_score(log_true, log_pred))

    # Sensitivity: the same metric with the defective rows removed. If these two differ
    # sharply, the headline log R2 is measuring the defects, not the model.
    if n_non_positive < len(y_pred) - 1:
        log_r2_excl = float(
            r2_score(log_true[~non_positive], log_pred[~non_positive])
        )
    else:
        log_r2_excl = float("nan")

    # How much of the log-space error budget the defective rows consume.
    log_residual = log_true - log_pred
    ss_res = float((log_residual**2).sum())
    ss_res_non_positive = float((log_residual[non_positive] ** 2).sum())
    non_positive_share_of_ss_res = (
        ss_res_non_positive / ss_res if ss_res > 0 else 0.0
    )

    # Multiplicative calibration: slope of log1p(true) on log1p(predicted). 1.0 is
    # unbiased on a log scale; <1.0 is shrinkage toward the middle of the range.
    calibration_slope = float(np.polyfit(log_pred, log_true, 1)[0])

    metrics = {
        "mae_php": mae,
        "rmse_php": rmse,
        "r2": r2,
        "mape_pct": mape_pct,
        "mae_pct_of_median": mae_pct_of_median,
        "log_space_r2": log_r2,
        "log_space_r2_excl_non_positive": log_r2_excl,
        "log_calibration_slope": calibration_slope,
        "n_non_positive_predictions": n_non_positive,
        "worst_prediction_php": worst_non_positive,
        "non_positive_share_of_log_ss_res": non_positive_share_of_ss_res,
        "n_test": int(len(y_true)),
        "test_median_php": median,
        "test_mean_php": float(y_true.mean()),
    }

    # VERIFICATION 4/5 (metric half): sane bounds.
    assert np.isfinite(
        [mae, rmse, r2, mape_pct, mae_pct_of_median, log_r2, calibration_slope]
    ).all(), "a metric came out non-finite"
    assert mae > 0, "MAE is zero, which means the predictions are not real"
    # Cauchy-Schwarz: ||e||_2 >= ||e||_1, so RMSE >= MAE always holds.
    assert rmse >= mae, f"RMSE {rmse:,.2f} < MAE {mae:,.2f}"
    assert r2 <= 1.0, f"R2 above 1.0 ({r2}) means the target leaked into the test set"
    return metrics


def baseline_r2(y_true: pd.Series, y_train: pd.Series) -> float:
    """R2 of always predicting the training mean - the bar a model must clear."""
    return float(r2_score(np.asarray(y_true), np.full(len(y_true), float(y_train.mean()))))


def check_fitted_pipeline(
    pipeline, x_frame, expected_rows: int, expected_width: int
) -> np.ndarray:
    """Assert the preprocessor hands the estimator a finite, all-numeric matrix.

    This is the guard that matters most in this environment: XGBoost 3.4.1 raises
    ``ValueError: DataFrame.dtypes for data must be int, float, bool or category`` if
    handed a pandas-3.0 ``str``-dtype DataFrame. The estimator must only ever see the
    preprocessor's numeric output.
    """
    matrix = pipeline.named_steps["preprocess"].transform(x_frame)
    assert matrix.shape[0] == expected_rows, (
        f"preprocessor returned {matrix.shape[0]} rows, expected {expected_rows}"
    )
    assert matrix.dtype.kind in "fiu", (
        f"preprocessor emitted dtype {matrix.dtype}; the estimator needs numeric input"
    )
    assert np.isfinite(matrix).all(), "preprocessor emitted NaN or inf"
    assert matrix.shape[1] == expected_width, (
        f"transformed width is {matrix.shape[1]}, expected {expected_width}"
    )
    return np.asarray(matrix, dtype="float32")


def fit_predict(name: str, x_train, y_train, x_eval, extra_numeric=()):
    """Fit one model end to end and return (pipeline, predictions)."""
    expected_input = [*FEATURES, *extra_numeric]
    assert list(x_train.columns) == expected_input, (
        f"expected columns {expected_input}, got {list(x_train.columns)}"
    )
    assert list(x_eval.columns) == expected_input, (
        f"train and evaluation frames disagree: {list(x_eval.columns)}"
    )

    pre = build_preprocessor(extra_numeric)
    # ColumnTransformer drops unnamed columns silently, so confirm the estimator really
    # receives every column that was handed to the pipeline.
    assert_preprocessor_consumes(pre, expected_input)

    pipeline = Pipeline(steps=[("preprocess", pre), ("model", build_estimator(name))])
    pipeline.fit(x_train, y_train)

    expected_width = EXPECTED_TRANSFORMED_WIDTH + len(extra_numeric)
    matrix = check_fitted_pipeline(pipeline, x_eval, len(x_eval), expected_width)
    predictions = pipeline.named_steps["model"].predict(matrix)
    assert np.isfinite(predictions).all(), f"{name} produced non-finite predictions"
    return pipeline, np.asarray(predictions, dtype="float64")


print("metric functions defined")

metric functions defined


## 6. Training the Models

A stratified 80/20 split on `property_type`, so each model sees the same four property
types in the same proportions. Stratifying on `property_type` rather than on price keeps
`lot` and `commercial` — a small slice of the data but a distinctive price regime —
represented on both sides.

In [7]:
def run_pipeline() -> dict:
    """The locked pipeline, start to finish. Called twice for the reproducibility check."""
    # VERIFICATION 3: 80/20 split with disjoint canonical_id sets.
    positions = np.arange(len(data))
    train_idx, test_idx = train_test_split(
        positions,
        test_size=TEST_SIZE,
        random_state=SPLIT_SEED,
        stratify=data["property_type"],
    )
    train_ids = set(data["canonical_id"].iloc[train_idx])
    test_ids = set(data["canonical_id"].iloc[test_idx])
    assert len(train_idx) + len(test_idx) == len(data), "split lost rows"
    assert not (train_ids & test_ids), "a canonical_id appears in both train and test"
    assert len(train_ids) == len(train_idx) and len(test_ids) == len(test_idx)
    assert round(len(test_idx) / len(data), 3) == TEST_SIZE, (
        f"test share is {len(test_idx) / len(data):.3f}, expected {TEST_SIZE}"
    )

    x_train, x_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    base = baseline_r2(y_test, y_train)

    print(f"train {len(train_idx):,} | test {len(test_idx):,} | "
          f"mean-baseline R2 {base:+.4f}")

    results = {}
    fitted = {}
    for name in MODEL_CONFIG:
        pipeline, predictions = fit_predict(name, x_train, y_train, x_test)
        metrics = evaluate(y_test, predictions)

        # VERIFICATION 5: R2 must beat the mean baseline.
        assert metrics["r2"] > base, (
            f"{name} R2 {metrics['r2']:.4f} does not beat the mean baseline {base:.4f}"
        )
        assert metrics["r2"] > 0.30, (
            f"{name} R2 {metrics['r2']:.4f} is implausibly low for this dataset; the "
            f"non-final smoke test reached 0.67, so something is broken"
        )
        results[name] = metrics
        fitted[name] = pipeline

        print(f"  {name:<26} MAE P{metrics['mae_php']:>13,.0f}  "
              f"RMSE P{metrics['rmse_php']:>13,.0f}  R2 {metrics['r2']:>7.4f}")

    # The model the Discussion quotes and the application loads.
    #
    # Stated plainly because it is easy to overclaim here: taking the maximum over two
    # models makes the quoted R2 a max-of-two and therefore mildly optimistic as an
    # estimate of performance on new data. It is a reporting choice between two fixed
    # configurations, not a selection procedure - no model was tuned, discarded or
    # re-fitted on the basis of these results - but Objective 2 is "Compare", so both
    # models' numbers are reported side by side and this one is not presented as
    # "the winner".
    reported = max(results, key=lambda k: results[k]["r2"])
    print(f"\nreported model: {reported} (R2 {results[reported]['r2']:.4f})")
    print(
        "note: this is the larger of two test R2 values, so it is mildly optimistic as a "
        "generalisation estimate. Both models are reported; neither was selected by search."
    )
    return {
        "metrics": results,
        "baseline_r2": base,
        "train_rows": int(len(train_idx)),
        "test_rows": int(len(test_idx)),
        "reported_model": reported,
        "fitted": fitted,
        "y_test": y_test,
    }


run = run_pipeline()

train 181,150 | test 45,288 | mean-baseline R2 -0.0000


  RandomForestRegression     MAE P   19,676,634  RMSE P   58,228,668  R2  0.6411


  XGBoostRegression          MAE P   21,751,582  RMSE P   60,102,168  R2  0.6177

reported model: RandomForestRegression (R2 0.6411)
note: this is the larger of two test R2 values, so it is mildly optimistic as a generalisation estimate. Both models are reported; neither was selected by search.


In [8]:
comparison = pd.DataFrame(run["metrics"]).T
comparison = comparison.rename(
    columns={
        "mae_php": "MAE (P)",
        "rmse_php": "RMSE (P)",
        "r2": "R2",
        "mape_pct": "MAPE (%)",
        "mae_pct_of_median": "MAE % of median",
        "log_space_r2": "log-space R2",
    }
)
print(f"Holdout: stratified 80/20 on property_type, seed {SPLIT_SEED}")
print(f"Mean baseline R2: {run['baseline_r2']:+.4f}\n")
print(
    comparison[["MAE (P)", "RMSE (P)", "R2", "MAPE (%)", "MAE % of median", "log-space R2"]]
    .to_string(float_format=lambda v: f"{v:,.4f}")
)

print("\nMultiplicative behaviour and prediction validity\n")
detail = pd.DataFrame(
    {
        "log-space R2": {k: v["log_space_r2"] for k, v in run["metrics"].items()},
        "log R2 excl. non-positive": {
            k: v["log_space_r2_excl_non_positive"] for k, v in run["metrics"].items()
        },
        "calibration slope": {
            k: v["log_calibration_slope"] for k, v in run["metrics"].items()
        },
        "non-positive preds": {
            k: v["n_non_positive_predictions"] for k, v in run["metrics"].items()
        },
        "worst prediction": {
            k: v["worst_prediction_php"] for k, v in run["metrics"].items()
        },
    }
)
print(detail.to_string(float_format=lambda v: f"{v:,.4f}"))

for name, m in run["metrics"].items():
    share = m["non_positive_share_of_log_ss_res"]
    print(
        f"\n{name}: {m['n_non_positive_predictions']} of {m['n_test']} predictions "
        f"({m['n_non_positive_predictions'] / m['n_test'] * 100:.2f}%) are zero or "
        f"negative, yet they account for {share * 100:.1f}% of the log-space error."
    )
print(
    "\nA negative price is not a small error, it is an impossible one, and log1p turns "
    "each into a residual of order 13-21. The log-space R2 above is therefore dominated "
    "by these rows for whichever model emits them. Both figures are reported so the "
    "effect can be judged rather than taken on trust."
)

Holdout: stratified 80/20 on property_type, seed 42
Mean baseline R2: -0.0000

                               MAE (P)        RMSE (P)     R2  MAPE (%)  MAE % of median  log-space R2
RandomForestRegression 19,676,634.2130 58,228,668.4311 0.6411  167.8327         161.6549        0.6427
XGBoostRegression      21,751,581.8578 60,102,167.8693 0.6177  183.7750         178.7018        0.0554

Multiplicative behaviour and prediction validity

                        log-space R2  log R2 excl. non-positive  calibration slope  non-positive preds  worst prediction
RandomForestRegression        0.6427                     0.6427             0.9421                   0      667,310.7906
XGBoostRegression             0.0554                     0.5712             0.5293                 195  -65,528,516.0000

RandomForestRegression: 0 of 45288 predictions (0.00%) are zero or negative, yet they account for 0.0% of the log-space error.

XGBoostRegression: 195 of 45288 predictions (0.43%) are zero or negat

## Visualizing the Results

Let's plot a direct comparison of the models so the differences in accuracy are easy to see at a glance.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as ticker

# Extract R2 and RMSE for plotting
models = list(run["metrics"].keys())
r2_scores = [run["metrics"][m]["r2"] for m in models]
rmse_scores = [run["metrics"][m]["rmse_php"] for m in models]

fig, ax1 = plt.subplots(figsize=(10, 5))

# Plot R2
sns.barplot(x=models, y=r2_scores, ax=ax1, palette=["#3498db", "#e74c3c"])
ax1.set_ylabel("R² Score (Higher is better)", fontweight='bold')
ax1.set_title("Model Comparison: Random Forest vs. XGBoost", fontsize=14, fontweight='bold')
ax1.set_ylim(0, max(r2_scores) * 1.15)

# Add R2 values on top of bars
for i, score in enumerate(r2_scores):
    ax1.text(i, score + 0.01, f"R²: {score:.4f}", ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

# Plot RMSE
fig, ax2 = plt.subplots(figsize=(10, 5))
sns.barplot(x=models, y=rmse_scores, ax=ax2, palette=["#2ecc71", "#f1c40f"])
ax2.set_ylabel("RMSE in PHP (Lower is better)", fontweight='bold')
ax2.set_title("Prediction Error (RMSE)", fontsize=14, fontweight='bold')
ax2.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, pos: f'₱{x/1e6:,.1f}M'))
ax2.set_ylim(0, max(rmse_scores) * 1.15)

# Add RMSE values on top of bars
for i, score in enumerate(rmse_scores):
    ax2.text(i, score + 1e6, f"₱{score:,.0f}", ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()


## 7. Testing on an Unseen City (Davao City Holdout)

A random split can flatter a spatial model: nearby listings share a city, and a city
learned from the training set is effectively memorised rather than generalised.

This check trains on everything **except Davao City** and tests on Davao City alone. The
model has never seen that city during training, so the target encoder has to place it
from its province and region alone. A large drop in R² here is expected and is exactly
what the Discussion should report — it bounds how far the model travels to a location
it was not trained on.

This is supplementary. The headline numbers in §6 are the stratified random split.

In [9]:
is_holdout_city = data["city"] == SPATIAL_HOLDOUT_CITY
n_holdout = int(is_holdout_city.sum())
print(f"spatial holdout: {SPATIAL_HOLDOUT_CITY} -> {n_holdout:,} rows "
      f"({n_holdout / len(data) * 100:.2f}% of the deduplicated data)")
assert n_holdout >= 1_000, (
    f"only {n_holdout} rows for {SPATIAL_HOLDOUT_CITY}; the holdout is too small to mean "
    f"anything. Check the city spelling."
)

x_train_s = X.loc[~is_holdout_city]
x_test_s = X.loc[is_holdout_city]
y_train_s = y.loc[~is_holdout_city]
y_test_s = y.loc[is_holdout_city]
assert len(x_train_s) + len(x_test_s) == len(X)
print(f"train {len(x_train_s):,} | test {len(x_test_s):,}\n")

spatial_metrics = {}
for name in MODEL_CONFIG:
    pipeline_s, predictions_s = fit_predict(name, x_train_s, y_train_s, x_test_s)
    metrics_s = evaluate(y_test_s, predictions_s)
    spatial_metrics[name] = metrics_s
    del pipeline_s, predictions_s
    gc.collect()
    print(f"  {name:<26} MAE P{metrics_s['mae_php']:>13,.0f}  "
          f"RMSE P{metrics_s['rmse_php']:>13,.0f}  R2 {metrics_s['r2']:>7.4f}")

spatial_frame = pd.DataFrame(spatial_metrics).T.rename(
    columns={"mae_php": "MAE (P)", "rmse_php": "RMSE (P)", "r2": "R2",
             "log_space_r2": "log-space R2"}
)
print()
print(spatial_frame[["MAE (P)", "RMSE (P)", "R2", "log-space R2"]].to_string(
    float_format=lambda v: f"{v:,.4f}"
))

spatial holdout: Davao City -> 3,886 rows (1.72% of the deduplicated data)
train 222,552 | test 3,886



  RandomForestRegression     MAE P   32,694,667  RMSE P   63,910,695  R2 -0.3631


  XGBoostRegression          MAE P   32,225,526  RMSE P   74,010,511  R2 -0.8280

                               MAE (P)        RMSE (P)      R2  log-space R2
RandomForestRegression 32,694,667.0798 63,910,695.4734 -0.3631       -1.2549
XGBoostRegression      32,225,526.4027 74,010,511.2052 -0.8280       -2.1677


## 8. Why We Dropped Price-Per-Sqm (Leakage Audit)

The dataset carries `price_per_sqm`, which is the asking price divided by floor area. It
is derived from the target, so using it as a feature is textbook target leakage: the
model is handed the answer and merely has to divide back.

This audit refits the same two models with that one column added, purely to quantify how
much the leak would inflate the metrics. It is reported so the exclusion is defensible
rather than asserted. The submitted model excludes it — see the leakage guard in §2.

Note that the leaked R² does **not** reach 1.0, and the cell measures why: `price_per_sqm`
is missing for part of the data, and a decision tree cannot multiply two features
together. Even with these limitations, we still see a massive ~17-point artificial jump in R² when this column is included, proving exactly why we excluded it from our final model!

In [10]:
positions = np.arange(len(data))
train_idx, test_idx = train_test_split(
    positions, test_size=TEST_SIZE, random_state=SPLIT_SEED,
    stratify=data["property_type"],
)
assert not (set(data["canonical_id"].iloc[train_idx])
            & set(data["canonical_id"].iloc[test_idx]))
assert len(train_idx) == run["train_rows"] and len(test_idx) == run["test_rows"], (
    "the audit split does not match the split used for the reported metrics"
)

# Rebuild the feature matrix WITH the leaked column. Same split, same config, so the only
# difference is the presence of price_per_sqm. extra_numeric routes it into the imputed
# numeric branch, which is what makes the audit measure the leak rather than nothing.
X_leaky = data[[*FEATURES, "price_per_sqm"]].copy()
condo = X_leaky["property_type"] == "condo"
X_leaky["lot_area_m2"] = np.where(
    condo & X_leaky["lot_area_m2"].isna() & X_leaky["sqm"].notna(),
    X_leaky["sqm"],
    X_leaky["lot_area_m2"],
)
print("AUDIT feature set adds exactly one column:", "price_per_sqm")

leaky_metrics = {}
for name in MODEL_CONFIG:
    pipeline_l, predictions_l = fit_predict(
        name,
        X_leaky.iloc[train_idx],
        y.iloc[train_idx],
        X_leaky.iloc[test_idx],
        extra_numeric=("price_per_sqm",),
    )
    metrics_l = evaluate(y.iloc[test_idx], predictions_l)
    leaky_metrics[name] = metrics_l
    del pipeline_l, predictions_l
    gc.collect()

audit_frame = pd.DataFrame(
    {
        "R2 (submitted)": {k: v["r2"] for k, v in run["metrics"].items()},
        "R2 (audit, leaked)": {k: v["r2"] for k, v in leaky_metrics.items()},
    }
)
audit_frame["inflation"] = audit_frame["R2 (audit, leaked)"] - audit_frame["R2 (submitted)"]
print()
print("AUDIT ONLY - not reportable as a result\n")
print(audit_frame.to_string(float_format=lambda v: f"{v:,.4f}"))

# First, establish that the leak is genuinely exact where it applies, so the audit is
# grounded in the data rather than in an assumption about it.
reconstructable = X_leaky["price_per_sqm"].notna() & X_leaky["sqm"].notna()
reconstructed = (
    X_leaky.loc[reconstructable, "price_per_sqm"] * X_leaky.loc[reconstructable, "sqm"]
)
max_gap = float((reconstructed - y[reconstructable]).abs().max())
max_relative_gap = float(
    ((reconstructed - y[reconstructable]).abs() / y[reconstructable]).max()
)
share = float(reconstructable.mean())
print(
    f"\nprice = price_per_sqm x sqm reproduces the target for {share:.2%} of rows "
    f"(max gap P{max_gap:,.2f}, {max_relative_gap:.2e} of the price)"
)
# Judged relatively, not absolutely: an absolute gap of a few thousand pesos is a rounding
# artifact on a listing of up to P1B, and would be meaningless against a P350,000 listing.
assert max_relative_gap < 1e-3, (
    f"price_per_sqm x sqm does not reproduce the target (worst relative gap "
    f"{max_relative_gap:.2e}); the column is not the leak this audit assumes it is"
)
print(
    "The audit R2 does not approach 1.0 for two measurable reasons:\n"
    f"  1. price_per_sqm is missing for {1 - share:.2%} of rows, so those targets are\n"
    "     unrecoverable and the imputer substitutes the column median.\n"
    "  2. Both estimators are decision-tree ensembles. A tree splits on one feature at a\n"
    "     time and so cannot form the product price_per_sqm x sqm, only a staircase\n"
    "     approximation of it. The leak is real but only partly exploitable by these\n"
    "     model families."
)

# The audit's job is to prove the column reaches the estimator, not to reach 1.0. A
# material rise in R2 is that proof; a rise of ~0 would mean the column was dropped.
for name in MODEL_CONFIG:
    inflation = leaky_metrics[name]["r2"] - run["metrics"][name]["r2"]
    assert inflation > 0.10, (
        f"{name} audit R2 rose by only {inflation:.4f}. The leaked column is probably "
        f"not reaching the estimator, so the audit would be measuring nothing."
    )
print("\nThe gap is the size of the leak. It is the reason price_per_sqm is excluded.")

AUDIT feature set adds exactly one column: price_per_sqm



AUDIT ONLY - not reportable as a result

                        R2 (submitted)  R2 (audit, leaked)  inflation
RandomForestRegression          0.6411              0.8138     0.1727
XGBoostRegression               0.6177              0.8102     0.1926

price = price_per_sqm x sqm reproduces the target for 85.01% of rows (max gap P1,174.30, 4.84e-06 of the price)
The audit R2 does not approach 1.0 for two measurable reasons:
  1. price_per_sqm is missing for 14.99% of rows, so those targets are
     unrecoverable and the imputer substitutes the column median.
  2. Both estimators are decision-tree ensembles. A tree splits on one feature at a
     time and so cannot form the product price_per_sqm x sqm, only a staircase
     approximation of it. The leak is real but only partly exploitable by these
     model families.

The gap is the size of the leak. It is the reason price_per_sqm is excluded.


## 9. Checking Reproducibility

`RESULTS_LOG.md` §6 requires the pipeline to be run **twice** with the metrics agreeing
to **6 decimal places**. Batch 0 measured a maximum per-prediction difference of
`1.164e-9` under `n_jobs=-1`, because thread scheduling changes the order of floating-point
additions. That is far below the sixth decimal of a peso-denominated metric, but it is
checked rather than assumed.

The second call re-runs the split and both model fits from scratch, drawing fresh splits
from the same seed and rebuilding the preprocessor, so a cached or stale estimator cannot
make this pass by accident. It reuses the deduplicated frame loaded in §1 rather than
re-reading the CSV, so this is a check on the *modelling* being deterministic, not on the
file read.

In [11]:
REPRO_DECIMALS = 6

first = {name: dict(m) for name, m in run["metrics"].items()}
gc.collect()
print("first run complete, releasing scratch objects before the second run\n")

run2 = run_pipeline()

print(f"\nreproducibility to {REPRO_DECIMALS} decimals")
keys = ["mae_php", "rmse_php", "r2", "mape_pct", "mae_pct_of_median", "log_space_r2"]
repro_rows = []
max_delta = 0.0
max_delta_raw = 0.0
for name in MODEL_CONFIG:
    for key in keys:
        a = round(first[name][key], REPRO_DECIMALS)
        b = round(run2["metrics"][name][key], REPRO_DECIMALS)
        delta = abs(a - b)
        raw_delta = abs(first[name][key] - run2["metrics"][name][key])
        max_delta = max(max_delta, delta)
        max_delta_raw = max(max_delta_raw, raw_delta)
        repro_rows.append(
            {"model": name, "metric": key, "run_1": a, "run_2": b, "abs_delta": delta}
        )
        assert a == b, (
            f"{name}.{key} is not reproducible to {REPRO_DECIMALS} decimals: "
            f"run 1 = {a}, run 2 = {b} (delta {delta:.3e})"
        )

print(pd.DataFrame(repro_rows).to_string(index=False, float_format=lambda v: f"{v:,.6f}"))
print(f"\nmax delta at {REPRO_DECIMALS} decimals: {max_delta:.3e}")
print(f"max delta before rounding          : {max_delta_raw:.3e}")
if max_delta_raw == 0.0:
    print(
        "Bit-identical this time. n_jobs=-1 sums partial results in thread-completion "
        "order, so repeated predict() calls can differ in the last few bits even on "
        "identical inputs; Batch 0 measured a 1.164e-9 spread that way. It did not "
        "appear in this run, and the unrounded figure above is reported precisely so "
        "that can be checked rather than taken on trust."
    )
else:
    print(
        f"Not bit-identical: {max_delta_raw:.3e}, from n_jobs=-1 accumulating partial "
        f"results in thread-completion order. It is far below the sixth decimal of a "
        f"peso, but it is reported unrounded rather than rounded away."
    )

reproducibility_agreed = max_delta == 0.0

del run2
gc.collect()

first run complete, releasing scratch objects before the second run



train 181,150 | test 45,288 | mean-baseline R2 -0.0000


  RandomForestRegression     MAE P   19,676,634  RMSE P   58,228,668  R2  0.6411


  XGBoostRegression          MAE P   21,751,582  RMSE P   60,102,168  R2  0.6177

reported model: RandomForestRegression (R2 0.6411)
note: this is the larger of two test R2 values, so it is mildly optimistic as a generalisation estimate. Both models are reported; neither was selected by search.

reproducibility to 6 decimals
                 model            metric             run_1             run_2  abs_delta
RandomForestRegression           mae_php 19,676,634.212978 19,676,634.212978   0.000000
RandomForestRegression          rmse_php 58,228,668.431080 58,228,668.431080   0.000000
RandomForestRegression                r2          0.641142          0.641142   0.000000
RandomForestRegression          mape_pct        167.832672        167.832672   0.000000
RandomForestRegression mae_pct_of_median        161.654898        161.654898   0.000000
RandomForestRegression      log_space_r2          0.642666          0.642666   0.000000
     XGBoostRegression           mae_php 21,751,581.85775

261

In [12]:
# VERIFICATION 6: provenance fields present.
manifest = json.loads(MANIFEST.read_text(encoding="utf-8"))
provenance = {
    "sha256": manifest["dataset"]["sha256"],
    "rows_raw": manifest["dataset"]["rows_raw"],
    "rows_deduplicated": manifest["dataset"]["rows_deduplicated"],
    "columns": manifest["dataset"]["columns_count"],
    "versions": manifest["environment"],
    "seeds": manifest["seeds"],
}
for field in ("sha256", "rows_raw", "rows_deduplicated", "columns", "versions", "seeds"):
    assert field in provenance and provenance[field], f"manifest field missing: {field}"
    print(f"  {field:<20} {provenance[field] if field != 'versions' else 'present'}")

# The dataset this run actually read must be the dataset the manifest describes.
assert provenance["rows_raw"] == EXPECTED_ROWS_RAW, (
    "manifest row count disagrees with the locked expected value"
)
assert provenance["rows_deduplicated"] == EXPECTED_ROWS_DEDUP
assert provenance["columns"] == EXPECTED_COLUMNS
print("\nprovenance OK")

  sha256               f474ea4bf0c48fd03945a855d51d7eeb4a9e7a6b080727a251367bfa603b1cca
  rows_raw             241496
  rows_deduplicated    226438
  columns              25
  versions             present
  seeds                {'global': 42, 'train_test_split': 42, 'target_encoder_cv': 'KFold(n_splits=5, shuffle=True, random_state=42)'}

provenance OK


In [13]:
report = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "generated_by": "notebook/price_prediction_pipeline.ipynb",
    "task": "IT20L Final Project - Task 3 Results & Discussion",
    "disclaimer": "Stratified random split figures are the reportable results. The "
                  "spatial holdout is supplementary. The leakage audit is AUDIT ONLY and "
                  "must not be reported as a result.",
    "dataset": provenance,
    "environment": {
        "python": platform.python_version(),
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "scikit_learn": sklearn.__version__,
        "xgboost": xgboost.__version__,
    },
    "split": {
        "strategy": "stratified 80/20",
        "stratify_on": "property_type",
        "test_size": TEST_SIZE,
        "random_state": SPLIT_SEED,
        "train_rows": run["train_rows"],
        "test_rows": run["test_rows"],
        "train_test_id_overlap": 0,
    },
    "mean_baseline_r2": run["baseline_r2"],
    "config": MODEL_CONFIG,
    "holdout": run["metrics"],
    "spatial_holdout": {
        "city": SPATIAL_HOLDOUT_CITY,
        "n_test": int(n_holdout),
        "metrics": spatial_metrics,
    },
    "leakage_audit": {
        "label": "AUDIT ONLY - NOT A RESULT",
        "added_feature": "price_per_sqm",
        "metrics": leaky_metrics,
        "r2_inflation": {
            name: leaky_metrics[name]["r2"] - run["metrics"][name]["r2"]
            for name in MODEL_CONFIG
        },
    },
    "reproducibility": {
        "runs": 2,
        "decimals_compared": REPRO_DECIMALS,
        "max_abs_delta": max_delta,
        "max_abs_delta_before_rounding": max_delta_raw,
        "agreed": reproducibility_agreed,
        "note": (
            "bit-identical on this run"
            if max_delta_raw == 0.0
            else "n_jobs=-1 accumulates in thread-completion order, so the unrounded "
                 "delta is non-zero but far below the sixth decimal of a peso"
        ),
    },
    "reported_model": run["reported_model"],
}

RESULTS.write_text(json.dumps(report, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(f"wrote {RESULTS} ({RESULTS.stat().st_size:,} bytes)")

print("\nreportable holdout metrics")
for name, m in run["metrics"].items():
    print(f"  {name:<26} MAE P{m['mae_php']:>13,.2f}  RMSE P{m['rmse_php']:>13,.2f}  "
          f"R2 {m['r2']:.6f}")

wrote C:\Users\Lethe\Documents\Workspaces\General\General - Obsidian\Projects\IT20L_2026-2027_1STSEM-1STTERM\Deliverables\FINAL_PROJECT_1\results\metrics.json (6,690 bytes)

reportable holdout metrics
  RandomForestRegression     MAE P19,676,634.21  RMSE P58,228,668.43  R2 0.641142
  XGBoostRegression          MAE P21,751,581.86  RMSE P60,102,167.87  R2 0.617678


In [14]:
# Persist the full Pipeline (preprocessor + model) for the Batch 4 application.
# The application loads this artifact and predicts. It never retrains.
# compress=3 is lossless: it shrinks a 662MB forest to ~206MB without touching a single
# prediction, and the artifact is gitignored, so it is regenerated rather than committed.
pipeline = run["fitted"][run["reported_model"]]
joblib.dump(pipeline, ARTIFACT, compress=3)
size_mb = ARTIFACT.stat().st_size / 1024 / 1024
print(f"wrote {ARTIFACT.name} ({size_mb:.2f} MB) -> {run['reported_model']}")

# Prove the artifact is self-contained: reload it and predict without any of the objects
# above being in scope, using one real row from the dataset.
reloaded = joblib.load(ARTIFACT)
sample = X.iloc[[0]]
reloaded_prediction = float(reloaded.predict(sample)[0])
assert np.isfinite(reloaded_prediction)
assert reloaded_prediction > 0, "reloaded artifact produced a non-positive price"
print(f"reload check: P{reloaded_prediction:,.2f} for one real row")
print(f"feature order recorded in the artifact: {list(reloaded.feature_names_in_)}")
print("\nartifact is loadable and predicts without retraining")

wrote price_pipeline.joblib (205.95 MB) -> RandomForestRegression


reload check: P274,374,651.98 for one real row
feature order recorded in the artifact: ['sqm', 'lot_area_m2', 'bedrooms', 'bathrooms', 'lat', 'lng', 'city', 'province', 'region', 'property_type']

artifact is loadable and predicts without retraining


## 10. What was verified

| # | Requirement | Result |
|---|---|---|
| 1 | Deduplicated row count == 226,438 | `assert` in `deduplicate` |
| 2 | No target leakage columns in the feature matrix | `assert` in `build_features` |
| 3 | Train/test == 80/20, disjoint `canonical_id` | `assert` in `run_pipeline` |
| 4 | All transformed features finite, numeric | `assert` in `check_fitted_pipeline` |
| 5 | Metrics sane; R² beats the mean baseline | `assert` in `evaluate` + `run_pipeline` |
| 6 | Provenance fields present | `assert` in the metrics cell |
| — | Metrics reproduce to 6 decimals across two runs | `assert` in the reproducibility cell |

Outputs:

- `results/metrics.json` — every number the Results & Discussion may quote
- `artifacts/price_pipeline.joblib` — the fitted pipeline the application loads